# 05 — GLP-1 utilization analysis

**Objective.** Import, validate and analyse a legitimately obtained local WIdO export.

**Business relevance.** Establish a transparent descriptive evidence base without merging incompatible populations or implying causality.

**Source and dataset ID.** Wissenschaftliches Institut der AOK (WIdO), PharMaAnalyst 2024 ATC A10BJ active-ingredient table (`WIDO-PHARMA-A10BJ-2024`).  
**Acquisition.** Manual CSV export from the interactive WIdO PharMaAnalyst portal; no API extraction.  
**Exact input.** Legitimately obtained local file `data/external/wirkst_export.csv`; expected header in `data/external/wido_expected_schema.csv`.  
**Original format.** Semicolon-delimited Windows-1252 CSV.  
**Transformation status.** The notebook validates schema, year, ATC codes, missing values and duplicate keys, then parses German-formatted numeric fields into an in-memory analytical table.  
**Outputs.** `images/glp1/glp1_prescriptions_by_ingredient_2024.png`, `images/glp1/glp1_costs_by_ingredient_2024.png`, and displayed aggregate results; restricted row-level output is not published.  
**Reproduction limitation.** Redistribution rights were not confirmed. Full reproduction requires an independently obtained legitimate export; prescriptions are not patients and indication is unavailable.  
**Period / granularity / units.** 2024; Germany × active ingredient; thousand prescriptions, thousand DDD, thousand EUR, EUR per prescription and shares.

Last updated: 2026-08-27

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data' / 'processed'
plt.style.use('seaborn-v0_8-whitegrid')

> **Restricted source.** Redistribution rights were not confirmed. The source export and row-level processed table are not included. This module is reproducible with legitimately obtained external data.

In [2]:
WIDO_LOCAL = ROOT / 'data' / 'external' / 'wirkst_export.csv'
WIDO_COLUMNS = ['Jahr','ATC-Code','Wirkstoff','Verordnungen in Tsd.','Tagesdosen in Tsd. DDD','Nettokosten in Tsd. Euro','Nettokosten je Verordnung','Nettokosten je Tagesdosis']
SUBSTANCES = {'A10BJ01':'Exenatide','A10BJ02':'Liraglutide','A10BJ05':'Dulaglutide','A10BJ06':'Semaglutide'}

def german_number(series):
    return pd.to_numeric(series.astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False), errors='raise')

def load_wido(path=WIDO_LOCAL):
    raw=pd.read_csv(path,sep=';',encoding='cp1252',dtype=str)
    missing=set(WIDO_COLUMNS)-set(raw.columns)
    if missing: raise ValueError(f'Missing required columns: {sorted(missing)}')
    raw=raw.loc[raw['ATC-Code'].isin(SUBSTANCES)].copy()
    if set(raw['ATC-Code']) != set(SUBSTANCES) or raw['ATC-Code'].duplicated().any():
        raise ValueError('Expected one 2024 row for each of the four recorded A10BJ codes')
    if raw[WIDO_COLUMNS].isna().any().any() or set(raw['Jahr']) != {'2024'}:
        raise ValueError('Unexpected missing values or period')
    out=pd.DataFrame({'year':raw['Jahr'].astype(int),'atc_code':raw['ATC-Code'],'active_ingredient':raw['ATC-Code'].map(SUBSTANCES)})
    mapping={'Verordnungen in Tsd.':'prescriptions_thousand','Tagesdosen in Tsd. DDD':'ddd_thousand','Nettokosten in Tsd. Euro':'net_cost_thousand_eur','Nettokosten je Verordnung':'net_cost_per_prescription_eur','Nettokosten je Tagesdosis':'net_cost_per_ddd_eur'}
    for source,target in mapping.items(): out[target]=german_number(raw[source]).astype(float)
    out['prescription_share']=out.prescriptions_thousand/out.prescriptions_thousand.sum()
    out['cost_share']=out.net_cost_thousand_eur/out.net_cost_thousand_eur.sum()
    return out.sort_values('atc_code').reset_index(drop=True)

if WIDO_LOCAL.exists():
    df=load_wido(); print('Authorized WIdO export loaded:',WIDO_LOCAL); display(df.head())
else:
    df=None
    print('Authorized WIdO data are not included. Export the 2024 A10BJ active-ingredient table from WIdO PharMaAnalyst and save it at: data/external/wirkst_export.csv')
    print('Expected schema: data/external/wido_expected_schema.csv')

Authorized WIdO data are not included. Export the 2024 A10BJ active-ingredient table from WIdO PharMaAnalyst and save it at: data/external/wirkst_export.csv
Expected schema: data/external/wido_expected_schema.csv


In [3]:
if df is not None:
    print('Shape:',df.shape); display(df.dtypes.to_frame('dtype')); display(df.describe().T)
    print('Missing:',int(df.isna().sum().sum()),'duplicates:',int(df.duplicated(['year','atc_code']).sum()))
    totals={'prescriptions':df.prescriptions_thousand.sum()*1000,'cost_eur':df.net_cost_thousand_eur.sum()*1000}
    totals['cost_per_prescription']=totals['cost_eur']/totals['prescriptions']
    display(pd.Series(totals))
    df.plot.bar(x='active_ingredient',y='prescriptions_thousand',legend=False,title='Recorded GKV prescriptions by active ingredient — 2024'); plt.ylabel('Thousand prescriptions'); plt.show()
else:
    print('Previously verified aggregate results: 2,674,000 prescriptions; EUR 582,169,100 costs; approximately EUR 218 per prescription; semaglutide + dulaglutide approximately 93% of prescriptions and 89% of costs.')

Previously verified aggregate results: 2,674,000 prescriptions; EUR 582,169,100 costs; approximately EUR 218 per prescription; semaglutide + dulaglutide approximately 93% of prescriptions and 89% of costs.


## Interpretation

The restricted 2024 analysis recorded four active ingredients. Prescriptions are not patients; indication is unavailable; private, self-pay and inpatient activity are excluded; tirzepatide is not in this A10BJ extract. Brand names are orientation examples only and must not be conflated with active ingredients or indications.

## Limitations, outputs and conclusion

The analysis preserves source periods, units and denominators; it does not impute, interpolate or claim causality. Outputs are the displayed summary tables and Matplotlib figures. The conclusion is descriptive and should be read with the source-specific limitations above.

In [4]:
if df is not None:
    IMAGE_DIR = ROOT / 'images' / 'glp1'; IMAGE_DIR.mkdir(parents=True, exist_ok=True)
    ax=df.plot.bar(x='active_ingredient',y='prescriptions_thousand',legend=False,title='Recorded GKV prescriptions by active ingredient — 2024',figsize=(8,5),color='#4472C4'); ax.set_ylabel('Thousand prescriptions'); ax.set_xlabel('Active ingredient'); plt.xticks(rotation=20); plt.tight_layout(); plt.savefig(IMAGE_DIR/'glp1_prescriptions_by_ingredient_2024.png',dpi=180); plt.show()
    ax=df.plot.bar(x='active_ingredient',y='net_cost_thousand_eur',legend=False,title='Reported GKV medicine costs by active ingredient — 2024',figsize=(8,5),color='#ED7D31'); ax.set_ylabel('Thousand EUR'); ax.set_xlabel('Active ingredient'); plt.xticks(rotation=20); plt.tight_layout(); plt.savefig(IMAGE_DIR/'glp1_costs_by_ingredient_2024.png',dpi=180); plt.show()
else:
    print('GLP-1 figure export requires the authorized local WIdO file described above.')


GLP-1 figure export requires the authorized local WIdO file described above.
